# Landmark-Pair Fingerprinting vs MinHash/SimHash for Near-Duplicate Detection

## Overview
This notebook demonstrates **landmark-pair fingerprinting** — a Shazam-inspired method for detecting near-duplicate text passages — against MinHash and SimHash baselines.

### Key Components
- **Landmark extraction**: Top-K salient tokens via sliding-window TF-IDF + non-maximum suppression
- **Fingerprinting**: Shazam-style anchor-target pairs within a lookahead window
- **Baselines**: MinHash (Jaccard and containment) and SimHash (64-bit)
- **Datasets**: GLUE MRPC (4,076 sentence pairs) + 1,100 synthetic structural-edit pairs

### Verdict
**DISCONFIRM** — Landmark-pair does NOT outperform MinHash containment on synthetic data. Positional offset (delta) is load-bearing but in a *negative* direction at sentence scale.

In [ ]:
import subprocess, sys
def _pip(*a): subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *a])

# Packages NOT pre-installed on Colab (always install everywhere)
_pip('datasketch>=1.0')
_pip('loguru>=0.7')
_pip('psutil>=6.0')

# Core packages (pre-installed on Colab, install locally to match Colab env)
if 'google.colab' not in sys.modules:
    _pip('numpy==2.0.2', 'scipy==1.16.3', 'scikit-learn==1.6.1', 'matplotlib==3.10.0')

In [ ]:
import json
import hashlib
import math
import time
import numpy as np
import matplotlib.pyplot as plt
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import average_precision_score, precision_recall_curve
from datasketch import MinHash
from scipy.stats import norm as scipy_norm

In [ ]:
GITHUB_DATA_URL = "https://raw.githubusercontent.com/AMGrobelnik/ai-invention-cb9424-landmark-pair-fingerprinting-for-text-cr/main/round-2/experiment-1/demo/mini_demo_data.json"
import os

def load_data():
    """Load demo data from GitHub URL or local file."""
    try:
        import urllib.request
        with urllib.request.urlopen(GITHUB_DATA_URL) as response:
            return json.loads(response.read().decode())
    except Exception as e:
        print(f"GitHub fetch failed ({e}), trying local file...")
    
    if os.path.exists("mini_demo_data.json"):
        with open("mini_demo_data.json") as f:
            return json.load(f)
    
    raise FileNotFoundError("Could not load mini_demo_data.json from GitHub or local path")

In [ ]:
data = load_data()
print(f"✓ Loaded data: {data['metadata']['objective']}")
print(f"  Datasets: {data['metadata']['datasets_evaluated']}")
print(f"  Total examples: {sum(len(d['examples']) for d in data['datasets'])}")

## Configuration

Tunable parameters for the landmark-pair method. Start with **minimal** values for quick testing, then scale up.

In [ ]:
# ── DEMO CONFIG ────────────────────────────────────────────────────────────
# Set these to MINIMAL values first, scale up after testing

# Landmark extraction
TOP_K = 5                    # top-k salient tokens (demo: 5, full: 15)
WINDOW_SIZE = 10             # TF-IDF local scoring window
NMS_RADIUS = 3               # non-maximum suppression radius (tokens)

# Fingerprinting
LOOKAHEAD = 10               # max distance between anchor-target pairs (demo: 10, full: 20)
DELTA_QUANT = 5              # quantize positional offset to nearest N tokens

# MinHash
SHINGLE_SIZE = 5             # character k-shingles
NUM_PERM = 128               # MinHash permutations

# SimHash
SIMHASH_BITS = 64            # SimHash bit width
SIMHASH_SEED = 1234

# Ablations
K_VALUES = [5, 10]           # ablation landmark density (demo: [5,10], full: [5,10,15,20,30])
W_VALUES = [10, 20]          # ablation lookahead window (demo: [10,20], full: [10,20,50,100])

print("✓ Config loaded:")
print(f"  TOP_K={TOP_K}, LOOKAHEAD={LOOKAHEAD}")
print(f"  Ablations: K={K_VALUES}, W={W_VALUES}")

## Landmark Extraction

Extract salient tokens using **sliding-window TF-IDF scoring** + **non-maximum suppression (NMS)**.

The key insight: score each token by its *local* TF within a window × its *global* IDF, then select the top-K non-overlapping tokens (within 3 positions of each other).

In [ ]:
def build_tfidf(corpus):
    """Fit TF-IDF vectorizer on corpus."""
    vec = TfidfVectorizer(analyzer="word", token_pattern=r"\b\w+\b", lowercase=True,
                          max_features=5000, sublinear_tf=True)  # demo: 5k vocab
    vec.fit(corpus)
    return vec


def extract_landmarks(text, vec, top_k=TOP_K):
    """
    Return top-k (position, token, tfidf_score) landmarks from text.
    Sliding-window: each token scored by its global IDF × local TF within a 10-word window.
    """
    vocab = vec.vocabulary_
    idf = vec.idf_
    words = text.lower().split()
    if not words:
        return []

    scores = []
    for i, w in enumerate(words):
        if w not in vocab:
            continue
        # local TF in window
        lo, hi = max(0, i - WINDOW_SIZE // 2), min(len(words), i + WINDOW_SIZE // 2 + 1)
        local_count = words[lo:hi].count(w)
        local_tf = 1 + math.log(local_count) if local_count > 0 else 0
        score = local_tf * idf[vocab[w]]
        scores.append((i, w, score))

    if not scores:
        return []

    # Non-maximum suppression: within every 3-position window keep best
    scores.sort(key=lambda x: -x[2])
    selected = []
    covered = set()
    for pos, tok, sc in scores:
        if any(abs(pos - c) < NMS_RADIUS for c in covered):
            continue
        selected.append((pos, tok, sc))
        covered.add(pos)
        if len(selected) >= top_k:
            break

    selected.sort(key=lambda x: x[0])  # sort by position
    return selected


print("✓ Landmark functions defined")

## Landmark-Pair Fingerprinting

**Core method**: Hash pairs of (anchor, target) landmarks within a lookahead window. Jaccard similarity over the sets of hashes.

In [ ]:
def _quantize(delta):
    """Quantize offset to nearest DELTA_QUANT tokens."""
    return (delta + DELTA_QUANT // 2) // DELTA_QUANT * DELTA_QUANT


def compute_fingerprint(landmarks, lookahead=LOOKAHEAD, use_delta=True):
    """Shazam-inspired: hash pairs of (anchor, target) landmarks within lookahead window."""
    fp = []
    for i, (pos_a, tok_a, _) in enumerate(landmarks):
        for j in range(i + 1, len(landmarks)):
            pos_t, tok_t, _ = landmarks[j]
            if pos_t - pos_a > lookahead:
                break
            delta = _quantize(pos_t - pos_a) if use_delta else 0
            # Deterministic hash using token strings + delta
            key = f"{tok_a}|{tok_t}|{delta}"
            h = int(hashlib.sha256(key.encode()).hexdigest()[:8], 16)
            fp.append(h)
    return frozenset(fp)


def fingerprint_similarity(fp1, fp2):
    """Jaccard over fingerprint hash sets."""
    if not fp1 and not fp2:
        return 1.0
    inter = len(fp1 & fp2)
    union = len(fp1 | fp2)
    return inter / union if union else 0.0


print("✓ Landmark-pair functions defined")

## Baseline Methods

Implement MinHash (Jaccard and containment) and SimHash for comparison.

In [ ]:
def shingle(text, k=SHINGLE_SIZE):
    """Character k-shingles."""
    text = text.lower().replace(" ", "_")
    return {text[i:i+k] for i in range(len(text) - k + 1)} if len(text) >= k else {text}


def make_minhash(text, num_perm=NUM_PERM):
    m = MinHash(num_perm=num_perm)
    for s in shingle(text):
        m.update(s.encode("utf-8"))
    return m


def minhash_jaccard(m1, m2):
    return m1.jaccard(m2)


def minhash_containment(m1, m2, size1, size2):
    """Estimate containment as |A∩B|/min(|A|,|B|)."""
    j = m1.jaccard(m2)
    if size1 == 0 or size2 == 0:
        return 0.0
    min_sz = min(size1, size2)
    max_sz = max(size1, size2)
    union_est = max_sz + min_sz - j * (max_sz + min_sz) / (1 + j) if (1 + j) > 0 else max_sz
    inter_est = j * union_est
    return min(inter_est / min_sz, 1.0) if min_sz > 0 else 0.0


# SimHash
_RNG_SIMHASH = np.random.RandomState(SIMHASH_SEED)

def _init_simhash_projections(n_features):
    return _RNG_SIMHASH.randn(SIMHASH_BITS, n_features).astype(np.float32)


def compute_simhash(tfidf_vec, projections):
    """Compute 64-bit SimHash from dense TF-IDF vector."""
    dots = projections @ tfidf_vec
    bits = (dots > 0).astype(np.uint8)
    result = 0
    for b in bits:
        result = (result << 1) | int(b)
    return result


def simhash_similarity(h1, h2):
    """Normalized hamming similarity (1 - hamming_distance/64)."""
    xor = h1 ^ h2
    hamming = bin(xor).count("1")
    return 1.0 - hamming / SIMHASH_BITS


print("✓ Baseline functions defined")

## Evaluation Metrics

Compute precision-recall curve, average precision, F1, and recall@precision≥0.90.

In [ ]:
def compute_metrics(scores, labels):
    """Compute PR curve, AP, F1, recall@prec90."""
    scores_arr = np.array(scores)
    labels_arr = np.array(labels)

    ap = float(average_precision_score(labels_arr, scores_arr))

    prec, rec, thresholds = precision_recall_curve(labels_arr, scores_arr)

    # recall@prec>=0.90
    recall_at_prec90 = 0.0
    threshold_at_prec90 = float(thresholds[-1]) if len(thresholds) else 1.0
    for p, r, t in zip(prec, rec, thresholds):
        if p >= 0.90 and r > recall_at_prec90:
            recall_at_prec90 = float(r)
            threshold_at_prec90 = float(t)

    # F1 optimal
    f1_vals = 2 * prec[:-1] * rec[:-1] / (prec[:-1] + rec[:-1] + 1e-10)
    best_f1_idx = int(np.argmax(f1_vals))
    f1_optimal = float(f1_vals[best_f1_idx])
    threshold_f1 = float(thresholds[best_f1_idx]) if best_f1_idx < len(thresholds) else 1.0

    # PR curve downsampled for display
    pr_curve = []
    step = max(1, len(thresholds) // 20)
    for i in range(0, len(thresholds), step):
        pr_curve.append([round(float(thresholds[i]), 4),
                         round(float(prec[i]), 4),
                         round(float(rec[i]), 4)])

    return {
        "auc_pr": round(ap, 4),
        "recall_at_prec90": round(recall_at_prec90, 4),
        "threshold_at_prec90": round(threshold_at_prec90, 4),
        "f1_optimal": round(f1_optimal, 4),
        "threshold_at_f1_optimal": round(threshold_f1, 4),
        "precision_recall_curve": pr_curve,
    }


print("✓ Metrics function defined")

## Main Processing Pipeline

Run all 5 methods (landmark-pair with/without delta, MinHash Jaccard/containment, SimHash) on a list of pairs.

In [ ]:
def process_pairs(pairs, vec, projections, top_k=TOP_K, lookahead=LOOKAHEAD):
    """
    Run all 5 methods on a list of pairs. Returns scores per method.
    """
    n = len(pairs)
    lm_scores = []
    lm_no_delta_scores = []
    mhj_scores = []
    mhc_scores = []
    sh_scores = []

    # Build TF-IDF sparse matrix for SimHash
    all_texts = []
    for p in pairs:
        # Extract sentence1 and sentence2 from input JSON
        if isinstance(p["input"], str):
            inp = json.loads(p["input"])
        else:
            inp = p["input"]
        all_texts.append(inp["sentence1"])
        all_texts.append(inp["sentence2"])

    print(f"  Transforming {len(all_texts)} texts for TF-IDF/SimHash")
    tfidf_matrix = vec.transform(all_texts)

    print(f"  Computing fingerprints for {n} pairs")
    for i, p in enumerate(pairs):
        # Extract sentence1 and sentence2 from input JSON
        if isinstance(p["input"], str):
            inp = json.loads(p["input"])
        else:
            inp = p["input"]
        s1 = inp["sentence1"]
        s2 = inp["sentence2"]
        idx1 = 2 * i
        idx2 = 2 * i + 1

        # Landmark-pair
        lm1 = extract_landmarks(s1, vec, top_k=top_k)
        lm2 = extract_landmarks(s2, vec, top_k=top_k)
        fp1 = compute_fingerprint(lm1, lookahead=lookahead, use_delta=True)
        fp2 = compute_fingerprint(lm2, lookahead=lookahead, use_delta=True)
        fp1_nd = compute_fingerprint(lm1, lookahead=lookahead, use_delta=False)
        fp2_nd = compute_fingerprint(lm2, lookahead=lookahead, use_delta=False)

        lm_scores.append(fingerprint_similarity(fp1, fp2))
        lm_no_delta_scores.append(fingerprint_similarity(fp1_nd, fp2_nd))

        # MinHash
        mh1 = make_minhash(s1)
        mh2 = make_minhash(s2)
        mhj_scores.append(minhash_jaccard(mh1, mh2))
        sh1 = shingle(s1)
        sh2 = shingle(s2)
        mhc_scores.append(minhash_containment(mh1, mh2, len(sh1), len(sh2)))

        # SimHash
        v1 = tfidf_matrix[idx1].toarray()[0].astype(np.float32)
        v2 = tfidf_matrix[idx2].toarray()[0].astype(np.float32)
        norm1, norm2 = np.linalg.norm(v1), np.linalg.norm(v2)
        if norm1 > 0: v1 /= norm1
        if norm2 > 0: v2 /= norm2
        h1 = compute_simhash(v1, projections)
        h2 = compute_simhash(v2, projections)
        sh_scores.append(simhash_similarity(h1, h2))

    labels = [int(p["output"]) for p in pairs]
    return {
        "landmark_pair": lm_scores,
        "landmark_pair_no_delta": lm_no_delta_scores,
        "minhash_jaccard": mhj_scores,
        "minhash_containment": mhc_scores,
        "simhash": sh_scores,
        "labels": labels,
    }


print("✓ Processing pipeline defined")

## Run Demo Experiment

Execute the full pipeline on the demo data.

In [ ]:
print("\n" + "="*70)
print("LANDMARK-PAIR DEMO EXPERIMENT")
print("="*70)

t_start = time.perf_counter()

# Extract pairs from datasets
all_pairs = []
for dataset in data['datasets']:
    all_pairs.extend(dataset['examples'])

print(f"\n[LOAD] {len(all_pairs)} total pairs from {len(data['datasets'])} datasets")

# Build TF-IDF corpus
print(f"\n[TF-IDF] Building TF-IDF corpus...")
all_texts = []
for p in all_pairs:
    if isinstance(p["input"], str):
        inp = json.loads(p["input"])
    else:
        inp = p["input"]
    all_texts.append(inp["sentence1"])
    all_texts.append(inp["sentence2"])

vec = build_tfidf(all_texts)
n_features = len(vec.vocabulary_)
print(f"  Vocab size: {n_features}")

# SimHash projections
projections = _init_simhash_projections(n_features)

# Process pairs
print(f"\n[PROCESS] Running all 5 methods...")
results = process_pairs(all_pairs, vec, projections, top_k=TOP_K, lookahead=LOOKAHEAD)

# Compute metrics
print(f"\n[METRICS] Computing metrics...")
labels = results["labels"]
metrics = {}
for method in ["landmark_pair", "landmark_pair_no_delta", "minhash_jaccard", "minhash_containment", "simhash"]:
    metrics[method] = compute_metrics(results[method], labels)

elapsed = round(time.perf_counter() - t_start, 1)
print(f"\n[DONE] Elapsed: {elapsed}s")

## Results & Analysis

Display key metrics and findings.

In [ ]:
print("\n" + "="*70)
print("RESULTS SUMMARY")
print("="*70)

# Summary table
print("\nRecall @ Precision ≥ 0.90 (Primary Metric):")
print("-" * 60)
for method, m in sorted(metrics.items()):
    recall = m["recall_at_prec90"]
    auc = m["auc_pr"]
    print(f"  {method:30s}  Recall={recall:.4f}  AUC_PR={auc:.4f}")

# Comparison: landmark vs MinHash containment
lm_recall = metrics["landmark_pair"]["recall_at_prec90"]
cont_recall = metrics["minhash_containment"]["recall_at_prec90"]
delta = lm_recall - cont_recall

print(f"\nLandmark-Pair vs MinHash Containment:")
print("-" * 60)
print(f"  Landmark-pair recall @ prec>=0.90:  {lm_recall:.4f}")
print(f"  MinHash containment recall:         {cont_recall:.4f}")
print(f"  Delta (LM - Containment):           {delta:+.4f}")
print(f"  Verdict: {'CONFIRM' if delta >= 0.10 else 'DISCONFIRM' if delta < -0.02 else 'PARTIAL'}")

# Ablation: positional offset (delta)
lm_with = metrics["landmark_pair"]["recall_at_prec90"]
lm_no = metrics["landmark_pair_no_delta"]["recall_at_prec90"]
delta_effect = lm_no - lm_with

print(f"\nPositional Offset Ablation:")
print("-" * 60)
print(f"  With delta (positional):    {lm_with:.4f}")
print(f"  Without delta:              {lm_no:.4f}")
print(f"  Effect:                     {delta_effect:+.4f}")
print(f"  Conclusion: Delta is {'HARMFUL' if delta_effect > 0.01 else 'NEUTRAL' if abs(delta_effect) <= 0.01 else 'HELPFUL'}")

print(f"\n{'='*70}")

## Visualization

Plot recall@precision≥0.90 across all 5 methods.

In [ ]:
# Visualization: recall@prec90 across methods
methods = ["landmark_pair", "landmark_pair_no_delta", "minhash_jaccard", "minhash_containment", "simhash"]
recalls = [metrics[m]["recall_at_prec90"] for m in methods]
aucs = [metrics[m]["auc_pr"] for m in methods]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Plot 1: Recall @ Precision >= 0.90
colors = ['#1f77b4', '#aec7e8', '#ff7f0e', '#2ca02c', '#d62728']
ax1.bar(range(len(methods)), recalls, color=colors, alpha=0.7, edgecolor='black', linewidth=1.5)
ax1.set_xticks(range(len(methods)))
ax1.set_xticklabels([m.replace('_', '\n') for m in methods], fontsize=9)
ax1.set_ylabel('Recall @ Precision ≥ 0.90', fontsize=11, fontweight='bold')
ax1.set_ylim([0, 1.0])
ax1.grid(axis='y', alpha=0.3)
ax1.set_title('Recall @ High Precision (Demo Data)', fontsize=12, fontweight='bold')
for i, v in enumerate(recalls):
    ax1.text(i, v + 0.02, f'{v:.3f}', ha='center', va='bottom', fontsize=9)

# Plot 2: Area Under PR Curve
ax2.bar(range(len(methods)), aucs, color=colors, alpha=0.7, edgecolor='black', linewidth=1.5)
ax2.set_xticks(range(len(methods)))
ax2.set_xticklabels([m.replace('_', '\n') for m in methods], fontsize=9)
ax2.set_ylabel('Average Precision (AUC)', fontsize=11, fontweight='bold')
ax2.set_ylim([0, 1.0])
ax2.grid(axis='y', alpha=0.3)
ax2.set_title('Area Under Precision-Recall Curve', fontsize=12, fontweight='bold')
for i, v in enumerate(aucs):
    ax2.text(i, v + 0.02, f'{v:.3f}', ha='center', va='bottom', fontsize=9)

plt.tight_layout()
plt.savefig('landmark_pair_demo_results.png', dpi=100, bbox_inches='tight')
plt.show()

print("✓ Visualization saved to landmark_pair_demo_results.png")

## Summary

### Key Findings

1. **Landmark-pair does NOT outperform MinHash containment** on synthetic structural-edit data (containment = 1.0 vs landmark-pair = 0.92 in full run).

2. **Positional offset (delta) is HARMFUL** — removing it actually improves recall on MRPC. The offset adds noise at sentence scale.

3. **MinHash containment is surprisingly effective** on synthetic data because shared filler text makes containment estimates very high.

4. **SimHash also strong** on synthetic data but weak on natural MRPC pairs.

### Tuning the Demo

To run on full data or with different parameters:
- Increase `TOP_K` from 5 to 15-20 for more landmarks
- Increase `LOOKAHEAD` from 10 to 20+ for wider context
- Adjust `K_VALUES` and `W_VALUES` for ablation studies
- Load more examples from `mini_demo_data.json` or the full dataset

**Verdict: DISCONFIRM** — The landmark-pair method does not represent an improvement over well-tuned MinHash approaches.